# 02 Data Cleaning

## Objective
Standardize station names and connection labels so that nodes and edges match correctly in the graph.

## Inputs
- `data/processed/tfl_stations.csv`
- `data/processed/tube_connections.csv`

## Outputs
- `data/processed/stations_clean.csv`
- `data/processed/connections_clean.csv`

## Why this step matters
Graph-based analysis depends on exact matching between station names in the node table and station names in the edge table. Even small naming differences can create isolated nodes and incorrect network fragmentation.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
BASE_DIR = Path("..")
PROCESSED_DIR = BASE_DIR / "data" / "processed"

stations = pd.read_csv(PROCESSED_DIR / "tfl_stations.csv")
connections = pd.read_csv(PROCESSED_DIR / "tube_connections.csv")

print("stations shape:", stations.shape)
print("connections shape:", connections.shape)
stations.head()

stations shape: (1751, 5)
connections shape: (754, 3)


,station_id,station_name,lat,lon,zone
0,0400ZZLUAMS0,Amersham Underground Station,51.674206,-0.607362,NaN
1,0400ZZLUCAL0,Chalfont & Latimer Underground Station,51.667915,-0.560616,NaN
2,0400ZZLUCAL1,Chalfont & Latimer Underground Station,51.668122,-0.560624,NaN
3,0400ZZLUCSM0,Chesham Underground Station,51.705227,-0.611113,NaN
4,2100ZZLUCXY0,Croxley Underground Station,51.647069,-0.441746,NaN


## Step 1: Define a common station-name cleaning function

This function standardizes station names across both datasets by:
- converting text to lowercase
- removing suffixes like `station` and `underground`
- removing apostrophes
- handling symbols like `&`
- trimming extra spaces

In [3]:
def clean_station_name(name):
    if pd.isna(name):
        return name

    name = str(name).lower().strip()
    name = name.replace(" station", "")
    name = name.replace(" underground", "")
    name = name.replace(" london underground ltd", "")
    name = name.replace("'", "")
    name = name.replace("&", "and")

    if "/" in name:
        name = name.split("/")[0].strip()

    name = " ".join(name.split())
    return name

In [4]:
stations["station_name_clean"] = stations["station_name"].apply(clean_station_name)
stations[["station_name", "station_name_clean"]].head(20)

,station_name,station_name_clean
0,Amersham Underground Station,amersham
1,Chalfont & Latimer Underground Station,chalfont and latimer
2,Chalfont & Latimer Underground Station,chalfont and latimer
3,Chesham Underground Station,chesham
4,Croxley Underground Station,croxley
5,Chorleywood Underground Station,chorleywood
6,Chorleywood Underground Station,chorleywood
7,Moor Park Underground Station,moor park
8,Moor Park Underground Station,moor park
9,Rickmansworth Underground Station,rickmansworth


## Step 2: Map route connection IDs to station names

The route file contains station IDs rather than final names. This step maps each connection ID to its station name.

In [5]:
id_to_name = stations.set_index("station_id")["station_name"].to_dict()

connections["from_name"] = connections["from_id"].map(id_to_name)
connections["to_name"] = connections["to_id"].map(id_to_name)

connections[["from_id", "from_name", "to_id", "to_name"]].head()

,from_id,from_name,to_id,to_name
0,940GZZLUEAC,Elephant & Castle Underground Station,940GZZLULBN,Lambeth North Underground Station
1,940GZZLULBN,Lambeth North Underground Station,940GZZLUWLO,Waterloo Underground Station
2,940GZZLUWLO,Waterloo Underground Station,940GZZLUEMB,Embankment Underground Station
3,940GZZLUEMB,Embankment Underground Station,940GZZLUCHX,Charing Cross Underground Station
4,940GZZLUCHX,Charing Cross Underground Station,940GZZLUPCC,Piccadilly Circus Underground Station


In [6]:
connections["from_name_clean"] = connections["from_name"].apply(clean_station_name)
connections["to_name_clean"] = connections["to_name"].apply(clean_station_name)

connections[["from_name", "from_name_clean", "to_name", "to_name_clean"]].head(20)

,from_name,from_name_clean,to_name,to_name_clean
0,Elephant & Castle Underground Station,elephant and castle,Lambeth North Underground Station,lambeth north
1,Lambeth North Underground Station,lambeth north,Waterloo Underground Station,waterloo
2,Waterloo Underground Station,waterloo,Embankment Underground Station,embankment
3,Embankment Underground Station,embankment,Charing Cross Underground Station,charing cross
4,Charing Cross Underground Station,charing cross,Piccadilly Circus Underground Station,piccadilly circus
5,Piccadilly Circus Underground Station,piccadilly circus,Oxford Circus Underground Station,oxford circus
6,Oxford Circus Underground Station,oxford circus,Regent's Park Underground Station,regents park
7,Regent's Park Underground Station,regents park,Baker Street Underground Station,baker street
8,Baker Street Underground Station,baker street,Marylebone Underground Station,marylebone
9,Marylebone Underground Station,marylebone,Edgware Road (Bakerloo) Underground Station,edgware road (bakerloo)


## Step 3: Check for naming mismatches

This step verifies whether cleaned connection names exist in the cleaned station name set.

In [7]:
station_name_set = set(stations["station_name_clean"])

from_not_in_stations = set(connections["from_name_clean"]) - station_name_set
to_not_in_stations = set(connections["to_name_clean"]) - station_name_set

print("from mismatches:", len(from_not_in_stations))
print("to mismatches:", len(to_not_in_stations))
print("sample from mismatches:", list(from_not_in_stations)[:20])
print("sample to mismatches:", list(to_not_in_stations)[:20])

from mismatches: 0
to mismatches: 0
sample from mismatches: []
sample to mismatches: []


In [8]:
stations_fixed = stations.copy()
stations_fixed["station_name"] = stations_fixed["station_name_clean"]

connections_fixed = connections.copy()
connections_fixed["from_name"] = connections_fixed["from_name_clean"]
connections_fixed["to_name"] = connections_fixed["to_name_clean"]

In [9]:
stations_fixed = stations_fixed.dropna(subset=["station_id", "station_name", "lat", "lon"])
connections_fixed = connections_fixed.dropna(subset=["from_name", "to_name"])

stations_fixed = stations_fixed.drop_duplicates(subset=["station_name"])
connections_fixed = connections_fixed.drop_duplicates(subset=["from_name", "to_name", "line"])

## Step 4: Save cleaned station and connection datasets

These files are the final network inputs used in graph construction and simulation.

In [10]:
stations_fixed.to_csv(PROCESSED_DIR / "stations_clean.csv", index=False)
connections_fixed.to_csv(PROCESSED_DIR / "connections_clean.csv", index=False)

print("Saved corrected stations_clean.csv")
print("Saved corrected connections_clean.csv")

Saved corrected stations_clean.csv
Saved corrected connections_clean.csv
